# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [8]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 2500 listings
        price  minimum_nights  number_of_reviews  availability_365  \
count  2500.0          2500.0             2500.0            2500.0   
mean    148.6            14.8              147.9             183.7   
std     110.9             8.4               86.3             105.5   
min      20.5             1.0                0.0               0.0   
25%      71.7             8.0               74.0              92.0   
50%     117.5            15.0              145.0             182.0   
75%     188.9            22.0              222.2             277.0   
max    1032.4            29.0              299.0             364.0   

       reviews_per_month  
count             2500.0  
mean                 2.0  
std                  2.0  
min                  0.0  
25%                  0.6  
50%                  1.4  
75%                  2.8  
max                 15.2  


In [9]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £373
                  count   mean   std   min    25%    50%    75%    max
room_type                                                             
Entire home/apt  1251.0  176.3  75.7  28.0  119.6  163.4  223.5  372.6
Private room      942.0   87.3  39.5  20.9   59.0   78.6  106.0  277.9
Shared room       182.0   46.3  14.1  20.5   36.8   44.1   54.3   92.8


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


# Task 1: Compare price distributions, excluding shared rooms.
room_order = ['Entire home/apt', 'Private room']
room_colors = {'Entire home/apt': '#2563EB', 'Private room': '#F59E0B'}
price_data = df_cap.loc[df_cap['room_type'].isin(room_order)].copy()
room_medians = price_data.groupby('room_type')['price'].median()

fig_task1 = px.histogram(
    price_data,
    x='price',
    color='room_type',
    barmode='overlay',
    opacity=0.6,
    nbins=40,
    category_orders={'room_type': room_order},
    color_discrete_map=room_colors,
    title='Entire-home listings are typically pricier, while both room types are right-skewed',
    labels={'price': 'Nightly price (£; capped at 95th percentile)', 'count': 'Number of listings'},
    template='plotly_white'
)

for room_type in room_order:
    median = room_medians[room_type]
    fig_task1.add_vline(
        x=median,
        line_color=room_colors[room_type],
        line_dash='dash',
        line_width=2,
        annotation_text=f'{room_type} median £{median:.0f}',
        annotation_position='top'
    )

fig_task1.add_annotation(
    x=1, y=-0.2, xref='paper', yref='paper', xanchor='right',
    text=f"Prices above £{p95:.0f} excluded (95th-percentile cap).",
    showarrow=False
)
fig_task1.update_layout(margin=dict(b=80), legend_title_text='Room type')
fig_task1.show()


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [7]:
import plotly.graph_objects as go

# Task 2: Compare review activity for listings with at least one review per month.
activity = df.loc[
    df['reviews_per_month'].notna() & (df['reviews_per_month'] > 0)
].copy()
borough_medians = activity.groupby('neighbourhood')['reviews_per_month'].median()
borough_order = borough_medians.sort_values(ascending=False).index.tolist()
busiest_boroughs = borough_order[:2]
overall_median = activity['reviews_per_month'].median()

highlight_colors = ['#0F766E', '#F59E0B']
highlight_map = dict(zip(busiest_boroughs, highlight_colors))
fig_task2 = go.Figure()
for borough in borough_order:
    borough_values = activity.loc[
        activity['neighbourhood'] == borough, 'reviews_per_month'
    ]
    color = highlight_map.get(borough, '#CBD5E1')
    fig_task2.add_trace(go.Box(
        x=borough_values,
        y=[borough] * len(borough_values),
        name=borough,
        orientation='h',
        boxpoints='outliers',
        marker=dict(color=color, size=5),
        line=dict(color=color),
        fillcolor=color,
        opacity=0.8,
        showlegend=False
    ))

fig_task2.add_vline(
    x=overall_median,
    line_color='#111827',
    line_dash='dash',
    annotation_text=f'Overall median: {overall_median:.2f} reviews/month',
    annotation_position='top'
)
fig_task2.update_layout(
    title=f"{busiest_boroughs[0]} and {busiest_boroughs[1]} have the highest median review activity",
    xaxis_title='Reviews per month (booking-activity proxy)',
    yaxis_title='London borough',
    yaxis=dict(categoryorder='array', categoryarray=borough_order, autorange='reversed'),
    template='plotly_white',
    margin=dict(l=150, t=100)
)
fig_task2.add_annotation(
    x=0, y=1.08, xref='paper', yref='paper', xanchor='left',
    text=f"Only listings with >0 reviews/month; highlighted: {busiest_boroughs[0]} and {busiest_boroughs[1]}.",
    showarrow=False
)
fig_task2.show()
